# **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
Already up to date.


In [10]:
# Compile Cython files
os.chdir(LOCAL_REPO_PATH)
!python run_compile_all_cython.py
os.chdir(WORKING_DIR)

run_compile_all_cython: Found 11 Cython files in 5 folders...
run_compile_all_cython: All files will be compiled using your current python environment: '/home/luigi/.venvs/recsys/bin/python'
Compiling [1/11]: MatrixFactorizationImpressions_Cython_Epoch.pyx... 
/home/luigi/RecSys/CythonCompiler/compile_script.py:37: SyntaxWarning: invalid escape sequence '\.'
  extensionName = re.sub("\.pyx", "", fileToCompile)
MatrixFactorizationImpressions_Cython_Epoch.c: In function ‘__pyx_f_43MatrixFactorizationImpressions_Cython_Epoch_32MatrixFactorization_Cython_Epoch_sampleBPR_Cython’:
MatrixFactorizationImpressions_Cython_Epoch.c:29667:17: warning: ‘__pyx_v_start_pos_impression_items’ may be used uninitialized []8;;https://gcc.gnu.org/onlinedocs/gcc-15.2.0/gcc/Warning-Options.html#index-Wmaybe-uninitialized-Wmaybe-uninitialized]8;;]
29667 |       __pyx_t_4 = (__pyx_v_start_pos_impression_items + __pyx_v_index);
      |       ~~~~~~~~~~^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Ma

In [11]:
if IS_COLAB or IS_KAGGLE:
    !pip install optuna

import optuna

In [12]:
import importlib
import scipy.sparse as sps
import pandas as pd
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import hyperparameter_tuning

Running on local — storage at: /home/luigi/RecSys


# **Load Data**

In [13]:
# Load datasets
URM_train = sps.load_npz(paths.URM_TRAIN)
URM_validation = sps.load_npz(paths.URM_VALIDATION)

In [14]:
def evaluate_recommender(recommender, at):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# **Hyperparameter search**

In [15]:
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender
SIMILARITY = "pearson"

In [16]:
def perform_optimization(similarity, n_trials):
    # Define objective function for hyperparameter tuning
    STUDY_NAME = UserKNNCFRecommender.RECOMMENDER_NAME + "_" + similarity

    def objective_function(optuna_trial: optuna.trial.Trial) -> float:
        recommender_instance = UserKNNCFRecommender(URM_train)
        recommender_instance.fit(
            similarity=similarity,
            topK=optuna_trial.suggest_int("topK", 10, 1500),
            shrink=optuna_trial.suggest_int("shrink", 0, 2000),
            normalize=optuna_trial.suggest_categorical("normalize", [True, False]),
            feature_weighting=optuna_trial.suggest_categorical("feature_weighting", ["BM25", "TF-IDF", "none"])
        )

        return evaluate_recommender(recommender_instance, at=20)

    # Perform hyperparameter tuning
    save_results, optuna_study = hyperparameter_tuning(
        objective_function,
        study_name=STUDY_NAME,
        n_trials=n_trials
    )

    return save_results, optuna_study

In [ ]:
fd_results, optuna_study = perform_optimization(SIMILARITY, 100)

[I 2025-11-08 20:30:42,752] Using an existing study with name 'UserKNNCFRecommender_pearson' instead of creating a new one.


  0%|          | 0/100 [00:00<?, ?it/s]

Similarity column 27095 (100.0%), 1988.02 column/sec. Elapsed time 13.63 sec
[I 2025-11-08 20:31:02,315] Trial 3 finished with value: 0.0032129050232470036 and parameters: {'topK': 1163, 'shrink': 273, 'normalize': True, 'feature_weighting': 'none'}. Best is trial 3 with value: 0.0032129050232470036.
Similarity column 27095 (100.0%), 1905.48 column/sec. Elapsed time 14.22 sec
[I 2025-11-08 20:31:23,428] Trial 4 finished with value: 0.05079134553670883 and parameters: {'topK': 263, 'shrink': 1981, 'normalize': True, 'feature_weighting': 'TF-IDF'}. Best is trial 4 with value: 0.05079134553670883.
Similarity column 27095 (100.0%), 1851.73 column/sec. Elapsed time 14.63 sec
[I 2025-11-08 20:31:46,750] Trial 5 finished with value: 0.07103192806243896 and parameters: {'topK': 706, 'shrink': 494, 'normalize': False, 'feature_weighting': 'BM25'}. Best is trial 5 with value: 0.07103192806243896.
Similarity column 27095 (100.0%), 1908.52 column/sec. Elapsed time 14.20 sec
[I 2025-11-08 20:32:07,

In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

# **Hyperparameter tuning**

In [ ]:
STUDY_NAME = UserKNNCFRecommender.RECOMMENDER_NAME + "_tuning_" + SIMILARITY

def pearson_tuning_function(optuna_trial: optuna.trial.Trial) -> float:
    recommender_instance = UserKNNCFRecommender(URM_train)
    recommender_instance.fit(
        similarity=SIMILARITY,
        topK=optuna_trial.suggest_int("topK", 0, 100),
        shrink=optuna_trial.suggest_int("shrink", 0, 100),
        normalize=True,
        feature_weighting="TF-IDF"
    )

    return evaluate_recommender(recommender_instance, at=20)

# Perform hyperparameter tuning
save_results, optuna_study = hyperparameter_tuning(
    pearson_tuning_function,
    study_name=STUDY_NAME,
    n_trials=20
)

In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
- ADD HERE